In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q seaborn scikit-learn tqdm

In [ ]:
import os, sys, time, random, shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# ════════════════════════════════════════════════════════════════
# 1. VERIFY GPU
# ════════════════════════════════════════════════════════════════
assert torch.cuda.is_available(), \
    "No GPU! Go to Runtime > Change runtime type > T4 GPU"
device = torch.device("cuda")
torch.backends.cudnn.benchmark = True  # speeds up fixed-size inputs

print("=" * 60)
print("  VITAMIN DEFICIENCY — COLAB TRAINING")
print("=" * 60)
print(f"  GPU    : {torch.cuda.get_device_name(0)}")
print(f"  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"  PyTorch: {torch.__version__}")
print("=" * 60)

# ════════════════════════════════════════════════════════════════
# 2. CONFIG
# ════════════════════════════════════════════════════════════════
DRIVE_DATA_DIR = "/content/drive/MyDrive/dataset_fixed"  # Drive source
LOCAL_DATA_DIR = "/content/dataset_fixed"                # Colab SSD (fast)
SAVE_DIR       = "/content/drive/MyDrive"
MODEL_NAME     = "best_vitamin_model.pth"

CLASS_MAP = {
    "Vitamin A deficiency":                       "Vitamin A deficiency",
    "Vitamin B-12 deficiency":                    "Vitamin B-12 deficiency",
    "Vitamin B complex deficiency":               "Vitamin B-12 deficiency",
    "Vitamin C deficiency":                       "Vitamin C deficiency",
    "Vitamin D deficiency":                       "Vitamin D deficiency",
    "zinc, iron, biotin, or protein deficiency":  "Zinc/Iron/Biotin deficiency",
}

IMG_SIZE       = 380
BATCH_SIZE     = 64
EPOCHS         = 50
PATIENCE       = 15
UNFREEZE_EPOCH = 15
LR_HEAD        = 5e-5
LR_BODY        = 5e-6
WEIGHT_DECAY   = 1e-4
WARMUP_EPOCHS  = 3
NUM_WORKERS    = 4

# ════════════════════════════════════════════════════════════════
# 3. COPY DATASET FROM DRIVE → LOCAL SSD
#    This is the KEY fix — Drive reads are 5-10x slower than SSD
#    First epoch was slow because it was reading from Drive directly
# ════════════════════════════════════════════════════════════════
if not Path(LOCAL_DATA_DIR).exists():
    print("\n[ Copying dataset from Drive to local SSD... ]")
    print("  This takes 3-5 minutes once, then training is fast.\n")
    t0 = time.time()
    shutil.copytree(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    elapsed = time.time() - t0
    print(f"\n✓ Dataset copied in {elapsed/60:.1f} min → {LOCAL_DATA_DIR}")
else:
    print(f"\n✓ Local dataset already exists: {LOCAL_DATA_DIR}")

DATA_DIR = LOCAL_DATA_DIR  # use fast local SSD from now on

# ════════════════════════════════════════════════════════════════
# 4. SEED
# ════════════════════════════════════════════════════════════════
def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

# ════════════════════════════════════════════════════════════════
# 5. DATASET CLASS
# ════════════════════════════════════════════════════════════════
class VitaminDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, label

# ════════════════════════════════════════════════════════════════
# 6. TRANSFORMS
# ════════════════════════════════════════════════════════════════
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.25),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.35, contrast=0.35,
                           saturation=0.25, hue=0.08),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ════════════════════════════════════════════════════════════════
# 7. LOAD DATA
# ════════════════════════════════════════════════════════════════
IMAGE_EXTS  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
class_names = sorted(set(CLASS_MAP.values()))
num_classes = len(class_names)
c2i         = {c: i for i, c in enumerate(class_names)}

samples = []
for src, dst in CLASS_MAP.items():
    d = Path(DATA_DIR) / src
    if not d.exists():
        print(f"  WARNING: not found: {d}")
        continue
    for f in d.iterdir():
        if f.suffix.lower() in IMAGE_EXTS:
            samples.append((str(f), c2i[dst]))

labels = np.array([s[1] for s in samples])
total  = len(samples)

print(f"\n✓ {num_classes} classes | {total} total images\n")
print(f"  {'Class':<35} {'Count':>6}")
print("  " + "-" * 43)
for i, cn in enumerate(class_names):
    cnt = (labels == i).sum()
    print(f"  {cn:<35} {cnt:>6}")

counts = np.array([(labels == i).sum() for i in range(num_classes)])
print(f"\n  Imbalance ratio: {counts.max()/counts.min():.1f}x")

spl = StratifiedShuffleSplit(1, test_size=0.2, random_state=42)
tri, vai = next(spl.split(np.zeros(total), labels))
tr_s = [samples[i] for i in tri]
va_s = [samples[i] for i in vai]
print(f"\n✓ Train: {len(tr_s)} | Val: {len(va_s)}")

train_loader = DataLoader(
    VitaminDataset(tr_s, train_tf),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True, drop_last=True
)
val_loader = DataLoader(
    VitaminDataset(va_s, val_tf),
    batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=True
)
print("✓ DataLoaders ready")

# ════════════════════════════════════════════════════════════════
# 8. CLASS WEIGHTS + CRITERION
# ════════════════════════════════════════════════════════════════
tl_arr = np.array([s[1] for s in tr_s])
cc     = np.bincount(tl_arr, minlength=num_classes).astype(float)
isq    = 1.0 / np.sqrt(cc)
cw     = torch.tensor(isq / isq.sum() * num_classes,
                      dtype=torch.float32).to(device)
print(f"\n  {'Class':<35} {'Weight':>6}")
print("  " + "-" * 43)
for cn, w in zip(class_names, cw.cpu().numpy()):
    print(f"  {cn:<35} {float(w):>6.3f}")

criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

# ════════════════════════════════════════════════════════════════
# 9. MODEL
# ════════════════════════════════════════════════════════════════
model = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
for p in model.features.parameters():
    p.requires_grad = False
inf = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.LayerNorm(inf), nn.Dropout(0.3),
    nn.Linear(inf, 512), nn.GELU(),
    nn.LayerNorm(512), nn.Dropout(0.15),
    nn.Linear(512, num_classes)
)
model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"\n✓ EfficientNet-B4 | {inf} → 512 → {num_classes}")
print(f"  Trainable: {trainable:,} / {total_p:,} params")

# ════════════════════════════════════════════════════════════════
# 10. OPTIMIZER + SCHEDULER + AMP
# ════════════════════════════════════════════════════════════════
head_params = list(model.classifier.parameters())
head_ids    = {id(p) for p in head_params}
body_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = torch.optim.AdamW([
    {"params": head_params, "lr": LR_HEAD,  "weight_decay": WEIGHT_DECAY},
    {"params": body_params, "lr": 0.0,      "weight_decay": WEIGHT_DECAY},
])

def lr_fn(ep):
    if ep < WARMUP_EPOCHS:
        return (ep + 1) / WARMUP_EPOCHS
    return 0.5 * (1 + np.cos(
        np.pi * (ep - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    ))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_fn)
scaler    = torch.amp.GradScaler("cuda")
print("✓ Optimizer + AMP scaler ready")

# ════════════════════════════════════════════════════════════════
# 11. GRADUAL UNFREEZE
# ════════════════════════════════════════════════════════════════
unfrozen_n = [0]

def gradual_unfreeze(ep):
    if ep < UNFREEZE_EPOCH:
        return
    blocks = list(model.features.children())
    n = min((ep - UNFREEZE_EPOCH) // 3 * 2 + 2, len(blocks))
    if n > unfrozen_n[0]:
        for block in blocks[-n:]:
            for p in block.parameters():
                p.requires_grad = True
        if optimizer.param_groups[1]["lr"] == 0.0:
            optimizer.param_groups[1]["lr"] = LR_BODY
        unfrozen_n[0] = n
        print(f"\n  → Unfroze top-{n} backbone blocks | LR-body: {LR_BODY:.1e}")

# ════════════════════════════════════════════════════════════════
# 12. MIXUP
# ════════════════════════════════════════════════════════════════
def mixup(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(crit, pred, ya, yb, lam):
    return lam * crit(pred, ya) + (1 - lam) * crit(pred, yb)

# ════════════════════════════════════════════════════════════════
# 13. EVALUATE
# ════════════════════════════════════════════════════════════════
def evaluate():
    model.eval()
    all_preds, all_labels = [], []
    correct = total_n = 0
    with torch.no_grad():
        for imgs, labs in tqdm(val_loader, desc="  Validating", leave=False):
            imgs, labs = imgs.to(device), labs.to(device)
            with torch.amp.autocast("cuda"):
                _, p = torch.max(model(imgs), 1)
            correct  += (p == labs).sum().item()
            total_n  += labs.size(0)
            all_preds.extend(p.cpu().numpy())
            all_labels.extend(labs.cpu().numpy())
    acc = 100 * correct / total_n
    print(classification_report(all_labels, all_preds,
                                  target_names=class_names, zero_division=0))
    return acc, all_labels, all_preds

# ════════════════════════════════════════════════════════════════
# 14. DASHBOARD
# ════════════════════════════════════════════════════════════════
def plot_dashboard(tr_a, va_a, losses, gts, preds):
    ep = range(1, len(tr_a) + 1)
    fig = plt.figure(figsize=(20, 12))
    fig.suptitle("Vitamin Deficiency — Final Results (T4 + EfficientNet-B4)",
                 fontsize=16, fontweight="bold")
    gs = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0:2])
    ax1.plot(ep, tr_a, label="Train", color="#2196F3", lw=2)
    ax1.plot(ep, va_a, label="Val",   color="#FF5722", lw=2)
    ax1.fill_between(ep, tr_a, va_a, alpha=0.08, color="gray")
    for y, col, lbl in [(70,"orange","70%"),(75,"green","75%"),
                        (80,"gold","80%"),(85,"purple","85%")]:
        ax1.axhline(y=y, color=col, linestyle="--", alpha=0.6, label=lbl)
    ax1.set_title("Training vs Validation Accuracy")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Accuracy (%)")
    ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3); ax1.set_ylim(25, 100)

    ax2 = fig.add_subplot(gs[0, 2])
    ax2.plot(ep, losses, color="#4CAF50", lw=2)
    ax2.set_title("Training Loss")
    ax2.set_xlabel("Epoch"); ax2.grid(True, alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 0])
    cm  = confusion_matrix(gts, preds)
    pca = cm.diagonal() / cm.sum(axis=1) * 100
    sh  = [c.replace(" deficiency","").replace("Vitamin ","Vit.")
             .replace("Zinc/Iron/Biotin","Zinc") for c in class_names]
    cols = ["#4CAF50" if a>=75 else "#FF9800" if a>=55 else "#F44336" for a in pca]
    bars = ax3.barh(sh, pca, color=cols)
    for bar, val in zip(bars, pca):
        ax3.text(bar.get_width()+1, bar.get_y()+bar.get_height()/2,
                 f"{val:.1f}%", va="center", fontsize=11, fontweight="bold")
    ax3.axvline(x=75, color="green", linestyle="--", alpha=0.6, label="75%")
    ax3.set_title("Per-Class Recall"); ax3.set_xlim(0, 118)
    ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3, axis="x")

    ax4 = fig.add_subplot(gs[1, 1:])
    cmn = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=sh, yticklabels=sh, ax=ax4,
                linewidths=0.5, vmin=0, vmax=1, annot_kws={"size": 12})
    ax4.set_title("Normalised Confusion Matrix")
    ax4.set_xlabel("Predicted"); ax4.set_ylabel("Actual")
    plt.setp(ax4.get_xticklabels(), rotation=25, ha="right", fontsize=10)
    plt.setp(ax4.get_yticklabels(), rotation=0,  fontsize=10)

    save_path = os.path.join(SAVE_DIR, "training_dashboard.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Dashboard saved to Drive: {save_path}")

# ════════════════════════════════════════════════════════════════
# 15. TRAINING LOOP
# ════════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print(f"  TRAINING START")
print(f"  Model   : EfficientNet-B4 | {IMG_SIZE}px | batch {BATCH_SIZE}")
print(f"  Epochs  : {EPOCHS} | Patience: {PATIENCE}")
print(f"  Frozen  : epochs 1–{UNFREEZE_EPOCH}")
print(f"  Unfreeze: epoch {UNFREEZE_EPOCH+1} onwards")
print(f"  AMP     : enabled (float16, ~2x faster)")
print(f"  Target  : 78–85% val accuracy")
print("=" * 60 + "\n")

best       = 0.0
ctr        = 0
tr_a       = []
va_a       = []
loss_h     = []
best_gts   = []
best_preds = []
start_time = time.time()

for ep in range(EPOCHS):
    ep_start = time.time()
    gradual_unfreeze(ep)
    model.train()
    rl = ok = tot = 0
    use_mx = (ep >= UNFREEZE_EPOCH)

    # tqdm progress bar for each epoch
    pbar = tqdm(train_loader,
                desc=f"Epoch {ep+1:02d}/{EPOCHS}",
                unit="batch",
                ncols=100)

    for imgs, labs in pbar:
        imgs, labs = imgs.to(device), labs.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            if use_mx and random.random() < 0.15:
                imgs, ya, yb, lam = mixup(imgs, labs)
                loss = mixup_loss(criterion, model(imgs), ya, yb, lam)
            else:
                out  = model(imgs)
                loss = criterion(out, labs)
                _, p = torch.max(out, 1)
                ok  += (p == labs).sum().item()
                tot += labs.size(0)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        scaler.step(optimizer)
        scaler.update()
        rl += loss.item()

        # update tqdm bar with live loss
        pbar.set_postfix({
            "loss": f"{rl/max(pbar.n,1):.4f}",
            "acc":  f"{100*ok/max(tot,1):.1f}%"
        })

    pbar.close()
    scheduler.step()

    tr_acc           = 100 * ok / max(tot, 1)
    val_acc, gts, preds = evaluate()
    avg_loss         = rl / len(train_loader)
    ep_time          = time.time() - ep_start
    total_elapsed    = time.time() - start_time

    tr_a.append(tr_acc)
    va_a.append(val_acc)
    loss_h.append(avg_loss)

    gap  = tr_acc - val_acc
    flag = " ⚠ OVERFIT" if gap > 12 else ""
    lr_h = optimizer.param_groups[0]["lr"]
    lr_b = optimizer.param_groups[1]["lr"]

    print(f"\nEpoch {ep+1:02d}/{EPOCHS} "
          f"| Loss: {avg_loss:.4f} "
          f"| Train: {tr_acc:.1f}% "
          f"| Val: {val_acc:.1f}% "
          f"| Gap: {gap:.1f}%{flag} "
          f"| {int(ep_time//60)}m{int(ep_time%60)}s "
          f"| Total: {int(total_elapsed//60)}m\n")

    if val_acc > best:
        best       = val_acc
        best_gts   = gts
        best_preds = preds

        # save locally + copy to Drive
        torch.save(model.state_dict(), MODEL_NAME)
        shutil.copy(MODEL_NAME, os.path.join(SAVE_DIR, MODEL_NAME))

        ctr = 0
        print(f"  ✓ NEW BEST: {best:.2f}% — saved to Drive ✓\n")
    else:
        ctr += 1
        if ctr >= PATIENCE:
            print(f"  Early stopping at epoch {ep+1}")
            break

# ════════════════════════════════════════════════════════════════
# 16. FINAL RESULTS
# ════════════════════════════════════════════════════════════════
total_time = time.time() - start_time
print("\n" + "=" * 60)
print(f"  TRAINING COMPLETE")
print(f"  Best accuracy : {best:.2f}%")
print(f"  Total time    : {int(total_time//3600)}h {int((total_time%3600)//60)}m")
print(f"  Model saved   : {SAVE_DIR}/{MODEL_NAME}")
print("=" * 60)

print("\n  PER-CLASS RESULTS:")
print("  " + "-" * 55)
cm  = confusion_matrix(best_gts, best_preds)
pca = cm.diagonal() / cm.sum(axis=1) * 100
for cn, recall in zip(class_names, pca):
    status = "✓ GOOD" if recall >= 75 else "~ OK" if recall >= 55 else "✗ LOW"
    bar    = "█" * int(recall // 5)
    print(f"  {cn:<35} {recall:5.1f}%  {bar}  {status}")
print(f"\n  Overall best accuracy: {best:.2f}%")

plot_dashboard(tr_a, va_a, loss_h, best_gts, best_preds)